# RAG Tutorial — Full Pipeline

**Goal:** Build a working Retrieval-Augmented Generation (RAG) system and understand how each stage affects the final answer.

This notebook follows the complete pipeline:

```text
PDF → Documents → Chunks → Embeddings → Vector Store
                         ↓
Question → Retrieval → Context → LLM → Answer
```

By the end of the tutorial, you should be able to:

- explain the main stages of a RAG pipeline;
- inspect retrieved chunks before generation;
- understand the roles of `top_k`, `chunk_size`, and `chunk_overlap`;
- evaluate generated answers against reference answers;
- run a small RAG experiment and interpret its results.

> **Recommended workflow:** Run the notebook from top to bottom. Sections 1–5 build the RAG system; Section 6 is the main hands-on testing activity; Section 7 shows how a framework can package the same workflow into a query engine.


# 1. Environment and API Configuration

Before building the RAG system, verify that the Python environment, required packages, LLM, and embedding API are working correctly.

## 1.1 Check the Python Environment and Dependencies

### Core tools

- **Python / Jupyter:** notebook runtime
- **LlamaIndex:** RAG orchestration
- **Chroma:** local vector store
- **Embedding model:** converts text into vectors
- **LLM:** generates answers from retrieved context
- **PyPDF:** extracts text from PDF files


In [ ]:
# Check the Python runtime and environment
import sys
import platform
from pathlib import Path

print("=" * 60)
print("Environment Check")
print("=" * 60)

print(f"Python       : {sys.version}")
print(f"Executable   : {sys.executable}")
print(f"Platform     : {platform.platform()}")
print(f"Working dir  : {Path.cwd().resolve()}")

assert sys.version_info >= (3, 9), "Python >= 3.9 is required."

print("\n✅ Python version check passed.")

# Check required packages
import importlib

packages = {
    "llama_index": "LlamaIndex",
    "chromadb": "Chroma",
    "openai": "OpenAI SDK",
    "pypdf": "PyPDF",
    "dotenv": "python-dotenv",
    "pandas": "Pandas",
    "sklearn": "scikit-learn",
}

print("=" * 60)
print("Package Check")
print("=" * 60)

for module, name in packages.items():
    try:
        importlib.import_module(module)
        print(f"✅ {name}")
    except ImportError as e:
        print(f"❌ {name}: {e}")

print("\n✅ Session 1 complete: environment and dependency checks finished.")


## 1.2 Configure the LLM and Embedding API

Keep the provider, model names, and API endpoint in one place. Switching providers should require configuration changes rather than a rewrite of the RAG pipeline.

Your `.env` file should contain:

```text
DASHSCOPE_API_KEY=your_api_key
DASHSCOPE_BASE_URL=your_base_URL
LLM_MODEL=qwen3.7-flash
EMBEDDING_MODEL=qwen3.7-text-embedding
```

Never add a real API key to the notebook or commit `.env` to GitHub.

> **Note:** Model Studio API keys are region-bound. Use an API key created for the region of your Base URL.


In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY") or os.getenv("dashscope_api_key")
DASHSCOPE_BASE_URL = (
    os.getenv("DASHSCOPE_BASE_URL")
    or os.getenv("dashscope_base_url")
    or "https://dashscope.aliyuncs.com/compatible-mode/v1"
)

LLM_MODEL = os.getenv("LLM_MODEL", "qwen3.7-flash")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "qwen3.7-text-embedding")

print("=" * 60)
print("Model Configuration")
print("=" * 60)

print(f"LLM model       : {LLM_MODEL}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Base URL        : {DASHSCOPE_BASE_URL}")

if DASHSCOPE_API_KEY:
    print("API key         : ✅ loaded")
else:
    print("API key         : ❌ missing")

print("\n✅ Model configuration successful.")

print("\n✅ Session 2 complete: model configuration loaded.")


Test both components before continuing:

1. Can the chat model generate a response?
2. Does the embedding model return a vector?

If either test fails, check the API key, endpoint, and model name before continuing.


In [ ]:
from openai import OpenAI

assert DASHSCOPE_API_KEY, (
    "DASHSCOPE_API_KEY not found. "
    "Please configure your .env file first."
)

client = OpenAI(
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
)

# ---- Test LLM ----
response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "user", "content": "Explain RAG in one sentence."}
    ],
)

print("LLM test:")
print(response.choices[0].message.content)

# ---- Test Embedding ----
embedding_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input="This is a test sentence for generating an embedding."
)

embedding = embedding_response.data[0].embedding

print("\nEmbedding test:")
print(f"Vector dimension = {len(embedding)}")
print(f"First 10 values  = {embedding[:10]}")

# 2. Indexing

Indexing converts the source PDFs into a searchable knowledge base.

The main steps are:

```text
PDF → Documents → Chunks → Embeddings → Chroma
```

## 2.1 Load the PDF

The notebook expects the following folder structure:

```text
data/
├── raw/        source PDFs
├── processed/  optional processed files
├── eval/       optional evaluation data
└── vector_db/  Chroma's persistent local database
```

Place the source PDFs in `data/raw/`. Chroma stores the vector database in `data/vector_db/`.


In [ ]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "eval"
VECTOR_DB_DIR = DATA_DIR / "vector_db"

for folder in [RAW_DIR, PROCESSED_DIR, EVAL_DIR, VECTOR_DB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(RAW_DIR.glob("*.pdf"))

print("=" * 60)
print("Session 3 — PDF Inventory")
print("=" * 60)

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {RAW_DIR.resolve()}. "
        "Please copy the course PDFs into data/raw/."
    )

for i, pdf in enumerate(pdf_files, start=1):
    print(f"{i:>2}. {pdf.name}  ({pdf.stat().st_size / 1024:.1f} KB)")

print(f"\nTotal PDFs: {len(pdf_files)}")

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {RAW_DIR.resolve()}.\n"
        "Please copy your course PDFs into data/raw/."
    )

reader = SimpleDirectoryReader(
    input_dir=str(RAW_DIR),
    # Set required_exts to restrict loading to specific file extensions.
    required_exts=[".pdf"],
    recursive=True,
)

documents = reader.load_data(show_progress=True)

print(f"\nLoaded document units: {len(documents)}")

print("\n✅ Session 3 complete: data folders and PDF inventory ready.")

### Inspect the PDF

For PDFs, a `Document` often represents one page or another parsed unit. It is **not** necessarily the final retrieval chunk.

Preserve document-level metadata such as file names and page numbers so that retrieved information can be traced back to its source.


In [ ]:
from IPython.display import display

# Inspect the first five document units
for i, doc in enumerate(documents[:5], start=1):
    print("=" * 80)
    print(f"Document unit #{i}")
    print(f"Metadata: {doc.metadata}")
    print(f"Text preview:\n{doc.text[:700]}")

# Run a basic data quality check
empty_docs = [i for i, doc in enumerate(documents) if not doc.text.strip()]
text_lengths = [len(doc.text) for doc in documents if doc.text.strip()]

print(f"Non-empty units : {len(text_lengths)}")
print(f"Empty units     : {len(empty_docs)}")

if text_lengths:
    print(f"Min chars      : {min(text_lengths):,}")
    print(f"Median chars   : {sorted(text_lengths)[len(text_lengths)//2]:,}")
    print(f"Max chars      : {max(text_lengths):,}")

## 2.2 Split the Documents into Chunks

The original PDF units are divided into smaller chunks for retrieval.

Two important parameters are:

- **`chunk_size`** — the approximate maximum size of each chunk.
- **`chunk_overlap`** — the amount of text shared between neighboring chunks.

These parameters affect what information can be retrieved together. We will investigate their effect more systematically in Section 6.


In [ ]:
from llama_index.core.node_parser import SentenceSplitter

CHUNK_SIZE = 512
CHUNK_OVERLAP = 100

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

nodes = splitter.get_nodes_from_documents(documents)

print("=" * 60)
print("Chunking Result")
print("=" * 60)

print(f"Original document units : {len(documents)}")
print(f"Generated chunks        : {len(nodes)}")
print(f"Chunk size              : {CHUNK_SIZE}")
print(f"Chunk overlap           : {CHUNK_OVERLAP}")

print("\n✅ Session 4 complete: chunks generated.")


## 2.3 Convert Chunks into Embeddings

Each text chunk is converted into an embedding vector.

Semantically similar text should be located closer together in the embedding space, allowing the retriever to find relevant chunks for a user question.


In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding(
    model_name=EMBEDDING_MODEL,
    api_key=DASHSCOPE_API_KEY,
    api_base=DASHSCOPE_BASE_URL,
    embed_batch_size=20,
)

print(f"Embedding model ready: {EMBEDDING_MODEL}")
print("\n✅ Session 5 complete: embedding model configured.")


### Create the Vector Index

Build a persistent vector index from the chunks using **Chroma** for storage and **LlamaIndex** to connect the chunks, embedding model, and vector store.

The collection is recreated when this cell is run so that the classroom workflow starts from a clean index.


In [ ]:
import chromadb
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

COLLECTION_NAME = "rag_course_demo"

chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DB_DIR)
)

# Rebuild collection for a clean classroom run
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    COLLECTION_NAME
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

print(f"Chroma collection: {COLLECTION_NAME}")
print(f"Persistent path  : {VECTOR_DB_DIR.resolve()}")

# Build the index from the chunks we explicitly created above.
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
    
)

print(f"Indexed chunks: {len(nodes)}")

# Record the chunking configuration used to build this index.
_INDEX_CHUNK_CONFIG = (CHUNK_SIZE, CHUNK_OVERLAP)

print("\n✅ Session 6 complete: vector index created.")


At this point, the knowledge base has been indexed:

```text
PDF → Documents → Chunks → Embeddings → Chroma
```

A RAG query adds the remaining stages:

```text
Question → Retrieval → Context Construction → Generation
```


# 3. Retrieval

Retrieval selects the chunks that are most relevant to a question.

## 3.1 Set `top_k`

`top_k` controls how many chunks are returned for each question.

- A **small** `top_k` gives the LLM less context.
- A **larger** `top_k` gives the LLM more context, but may also introduce irrelevant information.

We will vary this parameter later in Section 6.


In [ ]:
TOP_K = 4

retriever = index.as_retriever(
    similarity_top_k=TOP_K
)

print(f"Retriever configured with top_k = {TOP_K}")

print("\n✅ Session 7.1 complete: retriever is ready.")


## 3.2 Inspect Retrieval Results

Before asking the LLM to generate an answer, inspect the retrieved chunks.

Check:

- Are the returned chunks relevant?
- Does the highest-ranked chunk contain the answer?
- Do lower-ranked chunks add useful information or noise?
- Do the chunks come from the expected page and source?
- Does a chunk contain matching terms but fail to answer the question?

> **Key idea:** Retrieval can fail before the LLM is involved. Poorly retrieved context makes a grounded answer unlikely.


In [ ]:
import re
import html
from IPython.display import display, HTML

def get_file_name(node):
    metadata = node.metadata or {}
    return (
        metadata.get("file_name")
        or metadata.get("filename")
        or metadata.get("file_path")
        or "Unknown"
    )

def get_page_number(node):
    metadata = node.metadata or {}
    for key in ["page_label", "page_number", "page"]:
        value = metadata.get(key)
        if value is not None:
            try:
                return int(value)
            except (TypeError, ValueError):
                return value
    return None

def highlight_query_terms(text, query):
    """Highlight informative query terms in red."""
    stopwords = {
        "what", "is", "the", "a", "an", "of", "for", "on", "in", "to",
        "and", "each", "type", "be", "at", "as", "how", "many", "does",
        "will", "within", "according", "this", "document"
    }

    terms = [
        word.lower()
        for word in re.findall(r"\b[a-zA-Z0-9-]+\b", query)
        if word.lower() not in stopwords
    ]
    terms = sorted(set(terms), key=len, reverse=True)

    highlighted = html.escape(text)

    for term in terms:
        pattern = re.compile(
            rf"(?<![\w-])({re.escape(term)})(?![\w-])",
            re.IGNORECASE
        )
        highlighted = pattern.sub(
            r'<span style="color:#b00000; font-weight:bold;">\1</span>',
            highlighted
        )
    return highlighted

def inspect_retrieval(question, top_k=TOP_K, retrieved_nodes=None):

    # Run retrieval if results were not provided.
    if retrieved_nodes is None:
        retriever = index.as_retriever(
            similarity_top_k=top_k
        )
        retrieved_nodes = retriever.retrieve(question)

    print("=" * 80)
    print("RETRIEVED CHUNKS")
    print("=" * 80)

    for rank, item in enumerate(retrieved_nodes, start=1):

        node = item.node
        score = item.score

        file_name = get_file_name(node)
        page = get_page_number(node)

        print(f"\nRank {rank}")

        if score is not None:
            print(f"Score: {score:.4f}")
        else:
            print("Score: N/A")

        print(f"File : {file_name}")
        print(f"Page : {page}")

        print("\nChunk:")

        highlighted_text = highlight_query_terms(
            node.text,
            question
        )

        display(HTML(highlighted_text))

print("-" * 80)

print("\n✅ Retrieval inspection finished.")

question = (
    "What is the minimum sampling rate for on-site quality audit checks on each type of modular unit delivered to the building site?"
)

inspect_retrieval(question)

print("\n✅ Session 7.2 complete: retrieval results inspected.")

# 5. Generation

Retrieval tells us **what information the LLM should receive**. Generation determines **how the LLM uses that information to produce an answer**.

## 5.1 Configure the Chat Model

Configure the LLM used to generate the final answer.


In [ ]:
from llama_index.core.llms import ChatMessage
from llama_index.llms.openai_like import OpenAILike

llm = OpenAILike(
    model=LLM_MODEL,
    api_key=DASHSCOPE_API_KEY,
    api_base=DASHSCOPE_BASE_URL,
    temperature=0.2,
    is_chat_model=True,
)

print(f"LLM ready: {LLM_MODEL}")

print("\n✅ Session 8 complete: LLM is ready.")


## 5.2 System Prompt

The system prompt instructs the LLM to answer using the **retrieved context**, rather than relying only on its pretrained knowledge.

The prompt also tells the model what to do when the retrieved context does not contain enough information.


In [ ]:
SYSTEM_PROMPT = """You are a rigorous knowledge base question-answering assistant.

Please follow these rules:
1. Answer only based on the provided Context.
2. If the Context does not contain sufficient information to answer the question, explicitly state: “The knowledge base does not contain sufficient information.”
3. Do not add or infer facts that are not explicitly provided in the Context.
4. Keep your answers concise and clear whenever possible. 
5. At the end of your answer, provide the names of the source files used.
"""

def build_context(retrieved_nodes):
    context_blocks = []

    for i, item in enumerate(retrieved_nodes, start=1):
        node = item.node
        file_name = (
            node.metadata.get("file_name")
            or node.metadata.get("filename")
            or "unknown_file"
        )

        page = get_page_number(node)
        page_label = f", page {page}" if page is not None else ""

        context_blocks.append(
            f"[Source {i}] {file_name}{page_label}\n"
            f"{node.text}"
        )

    return "\n\n".join(context_blocks)


def build_rag_messages(question, retrieved_nodes):
    context = build_context(retrieved_nodes)

    user_prompt = f"""Please answer the following question.

Question:
{question}

Context:
{context}

Strictly base your answer on the provided Context.
"""

    return [
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=user_prompt),
    ]

def rag_query(question, top_k=TOP_K, verbose=True):

    # --------------------------------------------------
    # 1. Retrieval
    # --------------------------------------------------
    retriever = index.as_retriever(
        similarity_top_k=top_k
    )

    retrieved_nodes = retriever.retrieve(question)

    # --------------------------------------------------
    # 2. Build context
    # --------------------------------------------------
    messages = build_rag_messages(
        question,
        retrieved_nodes
    )

    # --------------------------------------------------
    # 3. Generate answer
    # --------------------------------------------------
    response = llm.chat(messages)

    answer = response.message.content

    # --------------------------------------------------
    # 4. Display
    # --------------------------------------------------
    if verbose:

        print("=" * 80)
        print("QUESTION")
        print("=" * 80)
        print(question)

        print("\nANSWER")
        print("=" * 80)
        print(answer)

        print("\nRETRIEVED CHUNKS")
        print("=" * 80)

        # Reuse retrieved_nodes to avoid running retrieval again.
        inspect_retrieval(
            question,
            retrieved_nodes=retrieved_nodes
        )

    # --------------------------------------------------
    # 5. Collect source information
    # --------------------------------------------------
    sources = []

    for rank, item in enumerate(retrieved_nodes, start=1):

        node = item.node

        sources.append({
            "rank": rank,
            "score": (
                float(item.score)
                if item.score is not None
                else None
            ),
            "file": get_file_name(node),
            "page": get_page_number(node),
            "node_id": node.node_id,
        })

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_nodes": retrieved_nodes,
    }

print("✅ Session 8.1 complete: RAG query function is ready.")

## 5.3 Generate an Answer

Use the same question from the retrieval experiment to see the complete RAG process:

```text
Question → Retrieve → Build Context → Generate Answer
```


In [ ]:
demo_question = (
    "What is the minimum sampling rate for on-site quality audit checks on each type of modular unit delivered to the building site?"
)

result = rag_query(
    demo_question,
    top_k=4
)

print("\n✅ Session 9 complete: first RAG query finished.")

# 6. Testing the RAG System

Now we test the complete RAG pipeline with a fixed set of questions.

The goal is not only to see whether the LLM can answer the questions. You will also investigate how three RAG parameters affect the final answer:

- **`top_k`** — how many chunks are retrieved for each question.
- **`chunk_size`** — the approximate size of each retrieval chunk.
- **`chunk_overlap`** — how much neighboring chunks overlap.

### Test workflow

1. Run **Test Set A** with the current settings.
2. Change the three parameters and rebuild the index when `chunk_size` or `chunk_overlap` changes.
3. Compare the answers with the reference answers in Section 6.2.
4. Use your observations to choose a final configuration.
5. Run **Test Set B** in Section 6.3 as a separate evaluation set.

> **Important:** `chunk_size` and `chunk_overlap` affect the indexed chunks, so changing either one requires rebuilding the vector index. Changing only `top_k` does not require re-indexing.


In [ ]:
import pandas as pd

# ============================================================
# RAG TEST SETTINGS
# ============================================================
# Students: change these values and rerun this cell.
TOP_K = 4
CHUNK_SIZE = 512
CHUNK_OVERLAP = 100

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("chunk_overlap must be smaller than chunk_size.")

print("=" * 60)
print("RAG Test Settings")
print("=" * 60)
print(f"top_k          : {TOP_K}")
print(f"chunk_size     : {CHUNK_SIZE}")
print(f"chunk_overlap  : {CHUNK_OVERLAP}")

current_chunk_config = (CHUNK_SIZE, CHUNK_OVERLAP)
previous_chunk_config = globals().get("_INDEX_CHUNK_CONFIG")

if previous_chunk_config != current_chunk_config:
    # chunk_size or chunk_overlap changed, so rebuild the vector index.
    splitter = SentenceSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    nodes = splitter.get_nodes_from_documents(documents)

    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = chroma_client.get_or_create_collection(COLLECTION_NAME)

    vector_store = ChromaVectorStore(
        chroma_collection=collection
    )

    storage_context = StorageContext.from_defaults(
        vector_store=vector_store
    )

    index = VectorStoreIndex(
        nodes,
        storage_context=storage_context,
        embed_model=embed_model,
        show_progress=True,
    )

    _INDEX_CHUNK_CONFIG = current_chunk_config

    print(f"\nIndexed chunks : {len(nodes)}")
    print("Index rebuilt because the chunking configuration changed.")
else:
    print("\nIndex reused: only top_k changed.")

retriever = index.as_retriever(similarity_top_k=TOP_K)

print("\n✅ Session 10.1 complete: RAG test configuration applied.")


### 6.1 Test Set A — Experiment with the RAG Configuration

Run the five questions below with your current settings. The result table records the RAG parameters used for each answer so that different runs can be compared.


In [ ]:
TEST_QUESTIONS_A = [
    "How many days does the Buildings Department (BD) take to provide a decision on a pre-submission enquiry?",
    "If the Buildings Department issues a letter of comments on a pre-acceptance application, within how many days must the Authorized Person (AP) and Registered Structural Engineer (RSE) clarify or respond to all comments before the application is considered abandoned?",
    "According to Table 1 of Appendix B, what is the minimum qualification required for a supervisor under the AP Stream, and how frequently must supervision be carried out?",
    "As an alternative arrangement to quality audits at the prefabrication factory, what is the minimum sampling rate for on-site quality audits of MiC modules delivered to the construction site?",
    "What qualification or certification must a factory have in order to manufacture Modular Integrated Construction (MiC) modules?",
]

test_results_A = []

for question in TEST_QUESTIONS_A:
    result = rag_query(question, top_k=TOP_K, verbose=False)

    test_results_A.append({
        "question": question,
        "answer": result["answer"],
        "top-k": TOP_K,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
    })

test_df_A = pd.DataFrame(test_results_A)

display(
    test_df_A.style.set_properties(
        subset=["question", "answer"],
        **{
            "white-space": "pre-wrap",
            "overflow-wrap": "anywhere",
            "min-width": "300px",
            "vertical-align": "top",
        },
    )
)

print("\n✅ Session 10.2 complete: Test Set A finished.")


## 6.2 Compare Test Set A with Reference Answers

Compare each generated answer with the corresponding reference answer.

The semantic similarity score below is based on cosine similarity between answer embeddings. **It is a review signal, not an automatic correctness score.** A high similarity does not guarantee that every important detail is correct, and a lower similarity does not necessarily mean that the answer is wrong.

When reviewing the results, pay particular attention to:

- whether the answer contains the required number, qualification, or condition;
- whether the retrieved context supports the answer;
- whether changing `top_k`, `chunk_size`, or `chunk_overlap` changes the answer.


In [ ]:
import numpy as np

REFERENCE_ANSWERS_A = [
    "The Buildings Department will provide a decision within 45 days.",
    "They must clarify or respond to all comments within 30 days from the date of the letter of comments.",
    "The minimum qualification is T3 level, equivalent to a Grade T3 Technically Competent Person as specified in the Code of Practice for Site Supervision. The required supervision frequency is once a week.",
    "The sampling rate must be at least 1% of each type of modular unit delivered to the construction site.",
    "The modules must be manufactured by a factory that holds a valid Full Certificate issued by the Building Technology Research Institute (BTRi) under the MiC Manufacturer Accreditation Scheme, or a factory that holds ISO 9001 or an equivalent quality assurance certification.",
]

if len(test_df_A) != len(REFERENCE_ANSWERS_A):
    raise ValueError("Reference answers must match Test Set A.")

answer_texts_A = test_df_A["answer"].fillna("").tolist()

all_embeddings = embed_model.get_text_embedding_batch(
    answer_texts_A + REFERENCE_ANSWERS_A
)

rag_embeddings_A = all_embeddings[:len(answer_texts_A)]
reference_embeddings_A = all_embeddings[len(answer_texts_A):]

semantic_similarities_A = []

for rag_embedding, reference_embedding in zip(
    rag_embeddings_A, reference_embeddings_A
):
    denominator = (
        np.linalg.norm(rag_embedding)
        * np.linalg.norm(reference_embedding)
    )

    similarity = (
        float(np.dot(rag_embedding, reference_embedding) / denominator)
        if denominator
        else np.nan
    )

    semantic_similarities_A.append(similarity)

answer_review_df_A = pd.DataFrame({
    "Question": test_df_A["question"],
    "RAG Answer": answer_texts_A,
    "Reference Answer": REFERENCE_ANSWERS_A,
    "Semantic Similarity": semantic_similarities_A,
    "top-k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
})

display(
    answer_review_df_A.style
    .format({"Semantic Similarity": "{:.3f}"})
    .set_properties(
        subset=["Question", "RAG Answer", "Reference Answer"],
        **{
            "white-space": "pre-wrap",
            "overflow-wrap": "anywhere",
            "min-width": "280px",
            "vertical-align": "top",
        },
    )
)

print(
    "\nReview the generated answers against the references and inspect the retrieved context when an answer looks incomplete."
)
print("✅ Session 10.3 complete: Test Set A reviewed.")


## 6.3 Test Set B — Evaluate Your Final Configuration

The five questions below were **not used in Test Set A**.

First, decide on the RAG configuration you want to keep. Then run this section **without changing the questions**.

This gives you a simple development/test workflow:

- **Test Set A:** explore and tune the RAG configuration.
- **Test Set B:** evaluate the final configuration on new questions.

The reference answers are provided only for evaluation after the answers have been generated.


In [ ]:
TEST_QUESTIONS_B = [
    "If a pre-acceptance application is rejected, how will any subsequent amended application be processed, and what is the time frame for the Buildings Department to make a decision?",
    "When a quality audit is conducted by videotelephony, what are the requirements for the colour, resolution, and storage medium of the recorded video?",
    "For the on-site quality audit of MiC components delivered to the construction site under the reinforcement works of the RSE Stream, how many locations must be measured using a covermeter, and how many readings must be taken at each location?",
    "After the initial site inspection of the prefabrication factory has been completed, how frequently must the AP, RSE, and AS of the RC conduct quality audit inspections of the prefabrication factory?",
    "If the AP/RSE adopts the alternative arrangement of conducting on-site quality audits of MiC components delivered to the construction site, how far in advance must they notify the Building Authority in writing?",
]

test_results_B = []

for question in TEST_QUESTIONS_B:
    result = rag_query(question, top_k=TOP_K, verbose=False)

    test_results_B.append({
        "question": question,
        "answer": result["answer"],
        "top-k": TOP_K,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
    })

test_df_B = pd.DataFrame(test_results_B)

display(
    test_df_B.style.set_properties(
        subset=["question", "answer"],
        **{
            "white-space": "pre-wrap",
            "overflow-wrap": "anywhere",
            "min-width": "300px",
            "vertical-align": "top",
        },
    )
)

print("\n✅ Session 10.4 complete: Test Set B finished.")


### Compare Test Set B with the Reference Answers

Now reveal the reference answers and review the final configuration on the unseen questions.


In [ ]:
REFERENCE_ANSWERS_B = [
    "Any subsequent amended application will be processed according to the time frame for a new application, and the Buildings Department will inform the applicant of its decision within 45 days.",
    "The video must be recorded in colour, have a resolution of at least 480p, and be stored on a non-rewritable DVD-ROM.",
    "Measurements must be taken at 6 locations, with 6 readings at each location within a 450 mm × 450 mm area.",
    "Quality audit inspections must be conducted at least once a month.",
    "They must notify the Building Authority one month before the commencement of production work at the prefabrication factory.",
]

if len(test_df_B) != len(REFERENCE_ANSWERS_B):
    raise ValueError("Reference answers must match Test Set B.")

answer_texts_B = test_df_B["answer"].fillna("").tolist()

all_embeddings_B = embed_model.get_text_embedding_batch(
    answer_texts_B + REFERENCE_ANSWERS_B
)

rag_embeddings_B = all_embeddings_B[:len(answer_texts_B)]
reference_embeddings_B = all_embeddings_B[len(answer_texts_B):]

semantic_similarities_B = []

for rag_embedding, reference_embedding in zip(
    rag_embeddings_B, reference_embeddings_B
):
    denominator = (
        np.linalg.norm(rag_embedding)
        * np.linalg.norm(reference_embedding)
    )

    similarity = (
        float(np.dot(rag_embedding, reference_embedding) / denominator)
        if denominator
        else np.nan
    )

    semantic_similarities_B.append(similarity)

answer_review_df_B = pd.DataFrame({
    "Question": test_df_B["question"],
    "RAG Answer": answer_texts_B,
    "Reference Answer": REFERENCE_ANSWERS_B,
    "Semantic Similarity": semantic_similarities_B,
    "top-k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
})

display(
    answer_review_df_B.style
    .format({"Semantic Similarity": "{:.3f}"})
    .set_properties(
        subset=["Question", "RAG Answer", "Reference Answer"],
        **{
            "white-space": "pre-wrap",
            "overflow-wrap": "anywhere",
            "min-width": "280px",
            "vertical-align": "top",
        },
    )
)

print("Review Test Set B as an unseen evaluation set rather than as another tuning set.")
print("✅ Session 10.5 complete: Test Set B reviewed.")


# 7. Extension — One-Step RAG with LlamaIndex

We built the RAG pipeline step by step so that each stage is visible.

LlamaIndex can package the same workflow into a query engine:

```python
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=TOP_K
)
```

Conceptually:

```text
Documents → Nodes → Index → Query Engine
```

This convenience API is useful once the underlying retrieval-and-generation workflow is understood.


In [ ]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=TOP_K,
)

framework_response = query_engine.query(
    "What are the minimum supervision frequencies for the AP, RSE and RC streams in Table 1?"
)

print(framework_response)

print("\n✅ Session 11 complete: framework-level RAG executed.")

# 8. Key Takeaways

**RAG = retrieve relevant external knowledge → augment the prompt → generate an answer.**

```text
PDF → Load → Chunk → Embed → Store → Retrieve → Augment → Generate
```

## Three Key Ideas

### 1. An LLM is not a knowledge base

The LLM generates answers; the vector store retrieves external knowledge.

### 2. Retrieval is not generation

```text
Retriever: What information should the LLM receive?
LLM:       How should I answer using that information?
```

### 3. RAG quality depends on the whole pipeline

```text
Documents → Chunking → Embeddings → Retrieval → Prompt → Generation
```

**RAG engineering is pipeline engineering.**


# Appendix — Environment Checklist and References

Before running the notebook:

- [ ] Python 3.9 or later and Jupyter are installed
- [ ] `DASHSCOPE_API_KEY` is configured
- [ ] `DASHSCOPE_BASE_URL` is configured or the notebook default is selected
- [ ] Source PDFs are placed in `data/raw/`
- [ ] The LLM and embedding API checks pass

## References

- Alibaba Cloud Model Studio — OpenAI-compatible Chat and Embedding APIs
- LlamaIndex documentation
- Chroma documentation

The provider and endpoint can change without changing the core RAG pipeline.
